In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from gpt_model3 import GPTModel3
from gpt_model1 import TokenDataset
import requests
from torchinfo import summary
from gpt_model5 import GPTModel5

# gpt 2 toenizer
from transformers import GPT2Tokenizer

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# tokenize the text
text = requests.get('https://www.gutenberg.org/files/35/35-0.txt').text

# text needs to be pytorch tensors
tokens = tokenizer.encode(text)
print(f'Variable "tokens" is type {type(tokens)}')

# convert to pytorch
tmTokens = torch.tensor( tokens )
print(f'Variable "tmTokens" is type {type(tmTokens)} and has {len(tmTokens)}')
print(tmTokens.shape)

Token indices sequence length is longer than the specified maximum sequence length for this model (48533 > 1024). Running this sequence through the model will result in indexing errors


Variable "tokens" is type <class 'list'>
Variable "tmTokens" is type <class 'torch.Tensor'> and has 48533
torch.Size([48533])


In [4]:
# Hyper parameters
seq_len = 1024 # aka context length
stride = 1
n_vocab = tokenizer.vocab_size
print("Vocab size = ", n_vocab)

# model hyperparameters
embed_dim = 768 # 64

batch_size = 32
num_transformerBlocks = 12
num_attn_heads = 12
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

Vocab size =  50257
mps


In [5]:
token_dataset = TokenDataset(tokenizer, text, seq_len, stride)
# print(len(token_dataset))
token_dataset[4]

dataloader = DataLoader(
                token_dataset,
                batch_size = batch_size,
                shuffle    = True,
                drop_last  = True
            )

# let's have a look at the indices
X,y = next(iter(dataloader))
print("X shape ", X.shape)
print("y shape ", y.shape)
print(tokenizer.decode(X.detach().numpy()[0]))
print(tokenizer.decode(y.detach().numpy()[0]))

X shape  torch.Size([32, 1024])
y shape  torch.Size([32, 1024])
 mere touch of the contrivance, the thing I had expected happened. The bronze panels suddenly slid up and struck the frame with a clang. I was in the dark—trapped. So the Morlocks thought. At that I chuckled gleefully.  “I could already hear their murmuring laughter as they came towards me. Very calmly I tried to strike the match. I had only to fix on the levers and depart then like a ghost. But I had overlooked one little thing. The matches were of that abominable kind that light only on the box.  “You may imagine how all my calm vanished. The little brutes were close upon me. One touched me. I made a sweeping blow in the dark at them with the levers, and began to scramble into the saddle of the machine. Then came one hand upon me and then another. Then I had simply to fight against their persistent fingers for my levers, and at the same time feel for the studs over which these fitted. One, indeed, they almost got away fr

In [6]:
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model5 = GPTModel5(vocab_size=n_vocab,
    embed_dim=embed_dim,
    context_len=seq_len,
    num_transformerBlocks=num_transformerBlocks,
    num_attn_heads=num_attn_heads,
    device=device)
# model5.to(device)

# model5.generate(torch.tensor([[1,2,3,4,5,6,7,8]]), T=1, gen_len=30)

In [12]:
summary(model5, input_data=X, col_names=['input_size', 'output_size', 'num_params'])

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
GPTModel5                                [32, 1024]                [32, 1024, 50257]         --
├─Embedding: 1-1                         [32, 1024]                [32, 1024, 768]           38,597,376
├─Embedding: 1-2                         [1024]                    [1024, 768]               786,432
├─Sequential: 1-3                        [32, 1024, 768]           [32, 1024, 768]           --
│    └─TransformerBlock: 2-1             [32, 1024, 768]           [32, 1024, 768]           --
│    │    └─LayerNorm: 3-1               [32, 1024, 768]           [32, 1024, 768]           1,536
│    │    └─MultiHeadAttention: 3-2      [32, 1024, 768]           [32, 1024, 768]           2,362,368
│    │    └─LayerNorm: 3-3               [32, 1024, 768]           [32, 1024, 768]           1,536
│    │    └─Sequential: 3-4              [32, 1024, 768]           [32, 1024, 768]           4,722,432
│ 

In [8]:
model5

GPTModel5(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (transformerBlocks): Sequential(
    (0): TransformerBlock(
      (layerNormAttn): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (W0): Linear(in_features=768, out_features=768, bias=True)
      )
      (layerNormMLP): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=768, out_features=3072, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=3072, out_features=768, bias=True)
      )
    )
    (1): TransformerBlock(
      (layerNormAttn): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (W0): Linear(in_features=768, out_features=768, bias=True)
      )
      (layerNormMLP): LayerNorm((768,), ep

In [10]:

X = X.to(device)
try:
    with torch.no_grad():
        logits = model5(X)
        print(logits.shape)
except RuntimeError as e:
    if "out of memory" in str(e):
        print(f"OOM at batch_size={batch_size}, try reducing it")

torch.Size([32, 1024, 50257])
